In [ ]:
from datasets import load_dataset
import numpy as np
import pandas as pd

from rds_chat_analysis import DATA_DIR, NOTEBOOK_DIR, REPO_ROOT
import shutil

In [ ]:
wildchat_dataset = load_dataset("allenai/WildChat-1M", split="train")
print("Wildchat schema:")
for k, v in wildchat_dataset[0].items():
    print(f"{k}: {type(v)}")

In [ ]:
np.random.seed(42)

# Split into 1000 mock logs, and use the rest for private logs
MAX_PRIVATE_LOGS = 10_000
MAX_MOCK_LOGS = 1000

all_indices = np.arange(len(wildchat_dataset))
np.random.shuffle(all_indices)

assert MAX_MOCK_LOGS <= len(
    all_indices
), f"Dataset is too small for {MAX_MOCK_LOGS} mock logs"

mock_logs = wildchat_dataset.select(all_indices[:MAX_MOCK_LOGS])
private_logs = wildchat_dataset.select(all_indices[MAX_MOCK_LOGS:])

if MAX_PRIVATE_LOGS is not None:
    private_logs = private_logs.select(range(MAX_PRIVATE_LOGS))

print(f"Mock logs: {len(mock_logs)}")
print(f"Private logs: {len(private_logs):,}")

In [ ]:
# Normalize to a dataframe with [id, conversation, metadata] schema


def clean_row(row):
    cleaned_conversation = []
    for turn in row["conversation"]:
        # Remove null bytes for postgres compatibility, normalize whitespaces
        content = (turn["content"] or "").replace("\x00", "").strip()
        content = " ".join(content.split())
        cleaned_conversation.append({"role": turn["role"], "content": content})
    return {
        "id": row["conversation_hash"],
        "conversation": cleaned_conversation,
        "metadata": {
            "timestamp": row["timestamp"],
            "model": row["model"],
        },
    }


mock_df = pd.DataFrame.from_records(
    [clean_row(row) for row in mock_logs],
)

private_df = pd.DataFrame.from_records(
    [clean_row(row) for row in private_logs],
)

In [ ]:
from syft_rds.orchestra import setup_rds_stack

key = "wildchat"
stack = setup_rds_stack(
    root_dir=REPO_ROOT / ".rds", key=key, log_level="DEBUG", reset=True
)

do_client = stack.do_rds_client
ds_client = stack.ds_rds_client

In [ ]:
DATASET_NAME = "Wildchat-10k"

local_data_dir = DATA_DIR / DATASET_NAME
private_dir = local_data_dir / "private"
mock_dir = local_data_dir / "mock"
markdown_path = local_data_dir / "README.md"

shutil.rmtree(local_data_dir, ignore_errors=True)

description = """
# Wildchat dataset 10k

This dataset contains 10,000 private logs and 100 mock logs from the Wildchat dataset, normalized to:
- `id`: conversation ID
- `conversation`: list of turns with `role` and `content`
- `metadata`: dictionary with `timestamp`, `model`

Example usage:
```python
import pandas as pd

# Load mock data
dataset = rds_client.dataset.get(name="Wildchat-10k")
mock_df = pd.read_parquet(dataset.mock_path / "data.parquet")
```
"""

private_dir.mkdir(parents=True, exist_ok=True)
mock_dir.mkdir(parents=True, exist_ok=True)

private_df.to_parquet(private_dir / "data.parquet", index=False)
mock_df.to_parquet(mock_dir / "data.parquet", index=False)
with open(markdown_path, "w") as f:
    f.write(description)

In [ ]:
# Move relevant .env files to mock and private directories
import shutil


MOCK_CREDENTIALS = NOTEBOOK_DIR / ".env.mock"
PRIVATE_CREDENTIALS = NOTEBOOK_DIR / ".env.private"

_ = shutil.copy(MOCK_CREDENTIALS, mock_dir / "credentials.env")
_ = shutil.copy(PRIVATE_CREDENTIALS, private_dir / "credentials.env")

print(f"Mock dir structure: {mock_dir}")
for file in mock_dir.iterdir():
    print(f"└──📄 {file.name}")
print(f"Private dir structure: {private_dir}")
for file in private_dir.iterdir():
    print(f"└──📄 {file.name}")

In [ ]:
wildchat_dataset = do_client.dataset.create(
    name=DATASET_NAME,
    path=private_dir,
    mock_path=mock_dir,
    summary="Wildchat dataset with 1000 private logs and 100 mock logs",
    description_path=markdown_path,
)

In [ ]:
wildchat_dataset.describe()